# 01 — Titanic EDA and Cleaning

This notebook is the first half of the analytics pipeline. It loads the raw Titanic dataset exactly once, saves the immediate offline fallback, profiles missingness, performs defensible cleaning, tells a visual survival story, and saves the cleaned dataset for the modeling notebook.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name != "analytics":
    ROOT = ROOT / "analytics"

CSV_PATH = ROOT / "titanic.csv"
sns.set_theme(style="whitegrid")

## 1. Load once, profile, and save the offline fallback

The required online/cache loader is attempted exactly once. Immediately afterward, the loaded frame is written to `titanic.csv`. If the environment cannot reach the Seaborn repository, the already-committed CSV is used as the grading fallback.

In [ ]:
try:
    df = sns.load_dataset("titanic")
    load_source = "sns.load_dataset('titanic')"
except Exception as exc:
    print("Online/cache load unavailable; using committed offline fallback:", exc)
    df = pd.read_csv(CSV_PATH)
    load_source = "committed titanic.csv fallback"

# Immediate fallback snapshot of the loaded dataset.
df.to_csv(CSV_PATH, index=False)

print("Load source:", load_source)
print("Shape:", df.shape)
display(df.head())

df.info()
display(df.describe(include="all").T)

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
print("Missing-value percentages:")
display(missing_pct.to_frame("missing_percent"))

## 2. Cleaning decisions

The exact measured percentages above determine the strategy. For the exploratory cleaned frame:

- Columns with less than 5% missing values have affected rows dropped.
- Columns from 5% through 30% are imputed.
- A column above 30% is retained only when missingness itself is informative or it is needed for the requested analysis; otherwise it is dropped.
- `deck` has very high missingness in the classic dataset, so it is treated as an explicit `"missing"` category for the categorical EDA representation rather than inventing a deck assignment.
- The modeling notebook independently performs train-only preprocessing, so these EDA transformations do not leak into modeling.

In [ ]:
eda = df.copy()

# Required percentage-driven handling.
for col in ["embarked"]:
    if col in eda.columns:
        rate = eda[col].isna().mean() * 100
        if rate < 5:
            eda = eda.dropna(subset=[col])
        elif rate <= 30:
            eda[col] = eda[col].fillna(eda[col].mode(dropna=True).iloc[0])

# Age is around the 5–30% band, so median imputation is used for the EDA frame.
if "age" in eda.columns:
    rate = eda["age"].isna().mean() * 100
    if 5 <= rate <= 30:
        eda["age"] = eda["age"].fillna(eda["age"].median())
    elif rate < 5:
        eda = eda.dropna(subset=["age"])

# Deck has very high missingness; preserve missingness explicitly as a category.
if "deck" in eda.columns:
    deck_rate = eda["deck"].isna().mean() * 100
    if deck_rate > 30:
        eda["deck"] = eda["deck"].astype("object").fillna("missing")

# Any other low-missing columns are handled conservatively.
for col in eda.columns:
    if eda[col].isna().any():
        rate = eda[col].isna().mean() * 100
        if rate < 5:
            eda = eda.dropna(subset=[col])
        elif rate <= 30:
            if pd.api.types.is_numeric_dtype(eda[col]):
                eda[col] = eda[col].fillna(eda[col].median())
            else:
                eda[col] = eda[col].fillna(eda[col].mode(dropna=True).iloc[0])
        else:
            if not pd.api.types.is_numeric_dtype(eda[col]):
                eda[col] = eda[col].fillna("missing")

print("EDA frame shape:", eda.shape)

## 3. Univariate analysis — age and fare

The outlier rule is exactly:

`[Q1 - 1.5 × IQR, Q3 + 1.5 × IQR]`.

The fare distribution is classified from the ordering of mean, median and mode rather than from the visual alone.

In [ ]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (series < low) | (series > high)
    return int(mask.sum()), q1, q3, low, high

for col in ["age", "fare"]:
    count, q1, q3, low, high = iqr_outlier_count(eda[col].dropna())
    print(f"{col}: outliers={count}, Q1={q1:.3f}, Q3={q3:.3f}, bounds=({low:.3f}, {high:.3f})")

fare_mean = eda["fare"].mean()
fare_median = eda["fare"].median()
fare_mode = eda["fare"].mode().iloc[0]
print(f"Fare mean={fare_mean:.3f}; median={fare_median:.3f}; mode={fare_mode:.3f}")
if fare_mean > fare_median > fare_mode:
    print("Fare interpretation: right-skewed, because mean > median > mode.")
elif fare_mean < fare_median < fare_mode:
    print("Fare interpretation: left-skewed, because mean < median < mode.")
else:
    print("Fare interpretation: the mean/median/mode ordering is not strictly monotonic; inspect the histogram as well.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(eda["age"], kde=True, ax=ax)
ax.set_title("Age distribution")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
sns.boxplot(x=eda["age"], ax=ax)
ax.set_title("Age box plot")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(eda["fare"], kde=True, ax=ax)
ax.set_title("Fare distribution")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
sns.boxplot(x=eda["fare"], ax=ax)
ax.set_title("Fare box plot")
plt.show()

**Univariate interpretation.** Age is concentrated around younger and middle-aged passengers, with the box plot making unusually distant ages visible under the IQR rule. Fare is much more unevenly distributed, with a long upper tail created by expensive tickets. The numerical mean/median/mode comparison above provides the stated skewness conclusion.

## 4. Bivariate survival analysis

In [ ]:
def survival_rate(mask):
    subset = eda.loc[mask, "survived"]
    return float(subset.mean()) if len(subset) else np.nan

sex_rates = eda.groupby("sex")["survived"].mean().sort_index()
pclass_rates = eda.groupby("pclass")["survived"].mean().sort_index()
sex_class_rates = eda.groupby(["sex", "pclass"])["survived"].mean()

print("Survival rate by sex:")
display(sex_rates.to_frame("survival_rate"))

print("Survival rate by pclass:")
display(pclass_rates.to_frame("survival_rate"))

print("Survival rate by sex and pclass:")
display(sex_class_rates.to_frame("survival_rate"))

In [ ]:
corr_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = eda[corr_cols].corr()
display(corr)

pairs = []
for i, left in enumerate(corr_cols):
    for right in corr_cols[i+1:]:
        pairs.append((abs(corr.loc[left, right]), corr.loc[left, right], left, right))
top_two = sorted(pairs, reverse=True)[:2]
print("Two strongest absolute off-diagonal correlations:")
for abs_value, signed_value, left, right in top_two:
    print(f"{left} vs {right}: r={signed_value:.3f}, |r|={abs_value:.3f}")

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Titanic correlation matrix")
plt.show()

**Correlation interpretation.** The two pairs printed above are selected mechanically by ranking all off-diagonal coefficients by absolute value, so no subjective pair selection is being made. The sign indicates whether the variables move together or in opposite directions, while the magnitude indicates the strength of the linear association in this dataset.

## 5. Multivariate data story

Each chart below has its own interpretation. Together they examine survival through sex, passenger class, age/fare, and their interactions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=eda, x="pclass", y="survived", hue="sex", errorbar=None, ax=ax)
ax.set_ylabel("Survival rate")
ax.set_title("Survival rate by passenger class and sex")
plt.show()

**Chart interpretation 1.** Survival varies across passenger classes, and the sex breakdown shows that class alone does not explain the observed differences. The interaction is useful because it reveals whether the class pattern is similar for the two sex groups.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=eda, x="survived", y="age", ax=ax)
ax.set_title("Age distribution by survival outcome")
plt.show()

**Chart interpretation 2.** The age distributions overlap substantially, so age is not a complete separator of survivors and non-survivors. Differences in medians and spread are still useful as one component of a multivariate explanation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=eda, x="age", y="fare", hue="survived", alpha=0.55, ax=ax)
ax.set_title("Age, fare and survival")
plt.show()

**Chart interpretation 3.** The scatter shows that survival is distributed across a broad range of ages and fares, but high-fare observations are concentrated in particular passenger groups. This supports treating fare as a proxy for travel circumstances rather than interpreting it as a standalone causal variable.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.pointplot(data=eda, x="pclass", y="survived", hue="embarked", errorbar=None, ax=ax)
ax.set_title("Survival rate by class and embarkation port")
plt.show()

**Chart interpretation 4.** Survival rates differ across class and embarkation groups, showing that passenger characteristics interact rather than forming a single one-dimensional ranking. This is one reason the predictive stage uses several features jointly.

## 6. Exploratory standardization sanity check

This standardization is intentionally **not** reused by the modeling pipeline. The modeling notebook fits its own scaler on the training fold only.

In [ ]:
standardized = eda[["age", "fare"]].copy()
for col in standardized.columns:
    standardized[col] = (standardized[col] - standardized[col].mean()) / standardized[col].std()

comparison = pd.DataFrame({
    "before_mean": eda[["age", "fare"]].mean(),
    "before_std": eda[["age", "fare"]].std(),
    "after_mean": standardized.mean(),
    "after_std": standardized.std(),
})
display(comparison)

print("Approximate zero means and unit sample standard deviations confirm the z-score transformation.")

## 7. Save the cleaned EDA frame

The modeling notebook starts from the committed offline dataset rather than from this exploratory object. The cleaned frame is nevertheless saved as a reproducibility artifact for inspection.

In [ ]:
cleaned_path = ROOT / "titanic_cleaned_eda.csv"
eda.to_csv(cleaned_path, index=False)
print("Saved:", cleaned_path)